# Phase 3: LSTM and comparison

Feed sequences of the last 7–30 days of price into a small LSTM to predict next-day price. Same train/val/test split and metrics as 01 and 02 for a fair comparison. Results table: Baselines vs Lag vs LSTM.

In [ ]:
# Colab: clone repo and install deps if not already there
import subprocess
import sys
from pathlib import Path
if 'google.colab' in sys.modules and not (Path('/content/crypto-price-prediction') / 'src').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/MOONx02/crypto-price-prediction.git', '/content/crypto-price-prediction'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/crypto-price-prediction/requirements.txt'], check=True)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

if 'google.colab' in sys.modules:
    ROOT = Path('/content/crypto-price-prediction')
else:
    ROOT = Path('.').resolve()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    elif ROOT.name != 'crypto-price-prediction':
        ROOT = ROOT / 'crypto-price-prediction'
sys.path.insert(0, str(ROOT))
from src.metrics import regression_metrics

## 1. Load data and time-based split (same as 01 and 02)

In [2]:
DATA_DIR = ROOT / 'data'
df = pd.read_parquet(DATA_DIR / 'BTC_USD_daily.parquet')
n = len(df)
train_end = int(0.70 * n)
val_end = int(0.85 * n)
price = df['price'].values.astype(np.float32)
print('Train 0 ->', train_end, '  Val', train_end, '->', val_end, '  Test', val_end, '->', n)

Train 0 -> 2338   Val 2338 -> 2839   Test 2839 -> 3340


## 2. Build sequences for LSTM (3.1)

**Input:** last `SEQ_LEN` days of price (one feature). **Target:** next-day price.

Shape: `(samples, timesteps, features)` = `(N, SEQ_LEN, 1)` for Keras LSTM. No future leakage: each sample uses only past data.

In [3]:
SEQ_LEN = 30  # same as N_LAGS in 02 for fair comparison

def build_sequences(price, train_end, val_end, seq_len):
    """Build X (N, seq_len, 1) and y (N,) for train / val / test using same split."""
    T = len(price)
    X_train_list, y_train_list = [], []
    for last in range(seq_len, train_end):
        X_train_list.append(price[last - seq_len : last])
        y_train_list.append(price[last])
    X_val_list, y_val_list = [], []
    for last in range(train_end, val_end):
        X_val_list.append(price[last - seq_len : last])
        y_val_list.append(price[last])
    X_test_list, y_test_list = [], []
    for last in range(val_end, T):
        X_test_list.append(price[last - seq_len : last])
        y_test_list.append(price[last])
    X_train = np.array(X_train_list, dtype=np.float32).reshape(-1, seq_len, 1)
    y_train = np.array(y_train_list, dtype=np.float32)
    X_val = np.array(X_val_list, dtype=np.float32).reshape(-1, seq_len, 1)
    y_val = np.array(y_val_list, dtype=np.float32)
    X_test = np.array(X_test_list, dtype=np.float32).reshape(-1, seq_len, 1)
    y_test = np.array(y_test_list, dtype=np.float32)
    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = build_sequences(price, train_end, val_end, SEQ_LEN)
print('X_train', X_train.shape, 'y_train', y_train.shape)
print('X_val  ', X_val.shape, 'y_val', y_val.shape)
print('X_test ', X_test.shape, 'y_test', y_test.shape)

X_train (2308, 30, 1) y_train (2308,)
X_val   (501, 30, 1) y_val (501,)
X_test  (501, 30, 1) y_test (501,)


## 3. LSTM model (3.2)

Small stack: 1–2 LSTM layers, then Dense(1) for regression. Same task: next-day price.

In [4]:
model = keras.Sequential([
    layers.Input(shape=(SEQ_LEN, 1)),
    layers.LSTM(32, return_sequences=True),
    layers.LSTM(16),
    layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 32)         │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 16)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,505 (29.32 KB)

 Trainable params: 7,505 (29.32 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=32,
    callbacks=[early],
    verbose=1
)

Epoch 1/80


In [ ]:
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.title('Loss (MSE)')
plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='train')
plt.plot(history.history['val_mae'], label='val')
plt.legend()
plt.title('MAE')
plt.tight_layout()
plt.show()

## 4. Evaluate on test set (3.3)

Same test period and metrics (MAE, RMSE, directional accuracy) as baselines and lag model.

In [ ]:
y_pred = model.predict(X_test, verbose=0).ravel()
m_lstm = regression_metrics(y_test, y_pred)
print('LSTM test set:')
print('  MAE:', m_lstm['mae'])
print('  RMSE:', m_lstm['rmse'])
print('  Dir.Acc:', f"{m_lstm['directional_accuracy']:.2%}")

## 5. Results table and summary (3.4)

Fill baseline and lag numbers from 01 and 02 (same test period). Short conclusion: which model wins on which metric.

In [ ]:
# Replace with your numbers from 01_data_and_baselines and 02_lag_model (same test set)
results = pd.DataFrame({
    'Model': ['Last value (baseline)', '7-day MA (baseline)', 'Lag + Ridge (02)', 'LSTM (this notebook)'],
    'MAE': [np.nan, np.nan, np.nan, m_lstm['mae']],
    'RMSE': [np.nan, np.nan, np.nan, m_lstm['rmse']],
    'Dir.Acc': [np.nan, np.nan, np.nan, m_lstm['directional_accuracy']]
})
results

In [ ]:
# After filling MAE/RMSE/Dir.Acc from 01 and 02, uncomment and run:
# results = results.fillna(...)  # or set values in the DataFrame above
# print(results.to_string(index=False))
# Summary: e.g. "LSTM improves MAE over baselines but may underperform the lag model; directional accuracy ..."

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(y_pred, y_test, alpha=0.5, s=15)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='y=x')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('LSTM: Predicted vs actual (test)')
ax.legend()
plt.tight_layout()
plt.show()